# {{PROJECT_NAME}} — 02 Exploratory Data Analysis

**Stage:** Data cleaning → EDA

Distributions first, relationships second. A correlation computed on a bimodal or heavily
skewed variable describes something other than what you think it does, which is why the
histograms come before the heatmap here.

> **This notebook is a stage template, not a mandatory step.** Use the notebooks your
> question needs and delete the rest — a descriptive project may never open
> `04_modeling.ipynb`, and a messy dataset may spend most of its life in `01`.
> See `CHECKLIST.md` for the full workflow.

In [ ]:
import sys, pathlib

ROOT = pathlib.Path.cwd()
ROOT = ROOT.parent if ROOT.name == 'notebooks' else ROOT
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.visualization.theme import PALETTE, save_fig, set_plot_style
set_plot_style()

# Point this at your file in data/raw/. Everything below follows from it.
DATASET = 'REPLACE_ME.csv'

In [ ]:
from src.data.loading import load_raw
from src.data.cleaning import snake_case_columns, strip_whitespace

df = load_raw(DATASET)

# Mechanical tidying — safe, reversible, no judgement involved.
df = snake_case_columns(strip_whitespace(df))

# Dataset-specific cleaning goes here, from what 01 found. Examples:
#   from src.data.cleaning import clean_number, coerce_types, standardize_category
#   df = coerce_types(df, {'salary': 'numeric', 'joined': 'datetime'})
#   df['country'] = df['country'].map(lambda v: standardize_category(v, {'AU': 'AUS'}))
df.head()

In [ ]:
from src.analysis.descriptive import describe_categorical, describe_numeric

describe_numeric(df)

In [ ]:
describe_categorical(df)

### Reading

_<Which variables are skewed? Which are near-constant? Does any mean sit far from its
median — and if so, is that skew or an outlier you decided to keep in 01?>_

In [ ]:
from src.visualization.plots import distribution_grid

fig, _ = distribution_grid(df)
save_fig(fig, '02_distributions')
plt.show()

## DECISION 4 — Pearson or Spearman?

The distributions above decide this, and the default everywhere else (Pearson) is often
the wrong tool.

- **Pearson** — linear relationships, interval data, sensitive to outliers and skew
- **Spearman** — monotonic relationships on ranks; robust to skew and outliers
- **Kendall** — rank-based, better for small n or many ties

- **Choice:** _<pearson / spearman / kendall>_ because _<what the histograms showed>_

In [ ]:
from src.analysis.correlation import correlation_matrix
from src.visualization.plots import correlation_heatmap

METHOD = 'pearson'   # <- set this from DECISION 4

corr = correlation_matrix(df, method=METHOD)
fig, _ = correlation_heatmap(corr, title=f'{METHOD.title()} correlation')
save_fig(fig, '02_correlation')
plt.show()

In [ ]:
from src.analysis.correlation import top_correlations_with
from src.visualization.plots import correlation_bars

TARGET = None   # <- the column your question is about

if TARGET:
    top = top_correlations_with(df, TARGET, method=METHOD)
    display(top)
    fig, _ = correlation_bars(top, title=f'Correlation with {TARGET}')
    save_fig(fig, '02_target_correlation')
    plt.show()
else:
    print('Set TARGET to the column your question is about.')

### Reading

_<Which relationships are strong enough to pursue, and which are strong but useless — a
proxy for the target, or collinear with a variable you already have? A ranked correlation
list is not a predictor shortlist.>_

In [ ]:
from src.analysis.descriptive import group_compare
from src.visualization.plots import category_counts

GROUP = None   # <- a categorical column worth splitting by, if you have one

if GROUP:
    fig, _ = category_counts(df, GROUP)
    save_fig(fig, '02_group_counts')
    plt.show()
    display(group_compare(df, GROUP))
else:
    print('Set GROUP if a categorical split is relevant to your question.')

## Findings so far

- _<what the data looks like, in two or three sentences>_
- _<the relationships worth taking to 03, and why>_
- _<what you have ruled out already>_

**Next stage:** _<03_analysis with a specific hypothesis / more cleaning / stop here if
description answers the question>_